# 🟩 Cell 1 — Install & Imports

In [1]:
# Modified Cell 1: added ML pipeline imports

import cv2
import numpy as np
import mediapipe as mp
import platform

from collections import deque
from sklearn.ensemble import RandomForestClassifier
import joblib

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 🟩 Cell 2 — Initialize MediaPipe

In [2]:
# Modified Cell 2: require MediaPipe's GPU delegate instead of silently using CPU.
GPU_REQUIRED = True
gpu_delegate = python.BaseOptions.Delegate.GPU if GPU_REQUIRED else python.BaseOptions.Delegate.CPU

try:
    base_options = python.BaseOptions(
        model_asset_path='hand_landmarker.task',
        delegate=gpu_delegate
    )
except Exception as exc:
    raise RuntimeError(
        f'Failed to configure the requested MediaPipe delegate on {platform.platform()}.'
    ) from exc

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

try:
    detector = vision.HandLandmarker.create_from_options(options)
    print(f'MediaPipe delegate requested: {gpu_delegate.name}')
except Exception as exc:
    if GPU_REQUIRED:
        raise RuntimeError(
            'GPU was required, but MediaPipe Tasks Python only supports the GPU delegate on Ubuntu platforms. '
            f'Current platform: {platform.platform()}. This notebook cannot force NVIDIA GPU execution from MediaPipe HandLandmarker on Windows.'
        ) from exc
    raise

RuntimeError: GPU was required, but MediaPipe Tasks Python only supports the GPU delegate on Ubuntu platforms. Current platform: Windows-10-10.0.26100-SP0. This notebook cannot force NVIDIA GPU execution from MediaPipe HandLandmarker on Windows.

# 🟩 Cell 3 — Servo State + System State

In [ ]:
# Modified Cell 3: added ML pipeline state
# Servo angles (0–180)
servo = {
    "S1": 90,  # gripper
    "S2": 90,
    "S3": 90,
    "S4": 90   # base
}

# System states
lock_mode = False
active_servo = None
cooldown = 0

# Previous values (for motion tracking)
prev_wrist_x = None
prev_depth = None
prev_thumb_angle = None
prev_pinch_dist = None

# ML gesture pipeline state
history = deque(maxlen=5)
gesture_buffer = deque(maxlen=5)

X_data = []
y_data = []

MODE = "collect"  # options: "collect", "train", "run"
model = None

# 🟩 Cell 4 — Gesture Detection Functions

In [ ]:
# Modified Cell 4: keep the lock helper and remove rule-based scoring.
# 🔒 Keep this for lock (unchanged)
def is_four_fingers(lm):
    tips = [8, 12, 16, 20]
    return sum(lm[tip].y < lm[tip - 2].y for tip in tips) == 4


# Modified Cell 4: rule-based score_* helpers were removed.
# The notebook now uses feature extraction, temporal context, and a classifier.

# 🟩 Cell 5 — Utility Functions

In [ ]:
# Modified Cell 5: utility helpers plus ML feature extraction.
def get_distance(p1, p2):
    return np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def get_thumb_angle(lm):
    dx = lm[4].x - lm[2].x
    dy = lm[4].y - lm[2].y
    return np.degrees(np.arctan2(dy, dx))

def extract_features(lm):
    return np.array([
        get_distance(lm[4], lm[8]),
        get_distance(lm[8], lm[12]),
        get_distance(lm[12], lm[16]),
        get_thumb_angle(lm),
        lm[0].z
    ])

def extract_temporal_features(history):
    velocities = []
    for i in range(1, len(history)):
        prev = history[i - 1]
        curr = history[i]
        v = get_distance(curr[8], prev[8])
        velocities.append(v)
    return np.mean(velocities) if velocities else 0

# 🟩 Cell 6 — Servo Control Functions

In [ ]:
#🔵 S4 — Horizontal Sweep (Base)
def control_s4(lm, frame_width):
    global prev_wrist_x

    x = lm[0].x * frame_width

    if prev_wrist_x is None:
        prev_wrist_x = x
        return

    delta = x - prev_wrist_x
    prev_wrist_x = x

    servo["S4"] += delta * 0.4
    servo["S4"] = np.clip(servo["S4"], 0, 180)

#🔴 S3 — Depth (Fist In/Out)
def control_s3(lm):
    global prev_depth

    depth = lm[0].z

    if prev_depth is None:
        prev_depth = depth
        return

    delta = depth - prev_depth
    prev_depth = depth

    servo["S3"] -= delta * 200
    servo["S3"] = np.clip(servo["S3"], 0, 180)

#🟡 S2 — Thumb Rotation
def control_s2(lm):
    global prev_thumb_angle

    angle = get_thumb_angle(lm)

    if prev_thumb_angle is None:
        prev_thumb_angle = angle
        return

    delta = angle - prev_thumb_angle
    prev_thumb_angle = angle

    servo["S2"] += delta
    servo["S2"] = np.clip(servo["S2"], 0, 180)

#🟢 S1 — Pinch (Gripper)
def control_s1(lm):
    global prev_pinch_dist

    dist = get_distance(lm[4], lm[8])

    if prev_pinch_dist is None:
        prev_pinch_dist = dist
        return

    delta = dist - prev_pinch_dist
    prev_pinch_dist = dist

    servo["S1"] += delta * 300
    servo["S1"] = np.clip(servo["S1"], 0, 180)

# 🟩 Cell 7 — Main Loop (FINAL STATE MACHINE)

In [ ]:
def draw_landmarks(frame, landmarks):
    h, w, _ = frame.shape

    # Draw points
    for lm in landmarks:
        cx, cy = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (cx, cy), 5, (0, 255, 255), -1)

    # Draw connections (hand skeleton)
    connections = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20)
    ]

    for c in connections:
        x1, y1 = int(landmarks[c[0]].x * w), int(landmarks[c[0]].y * h)
        x2, y2 = int(landmarks[c[1]].x * w), int(landmarks[c[1]].y * h)

        cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

In [ ]:
# Modified Cell 8: load the classifier before starting the camera loop.
cap = cv2.VideoCapture(0)

if MODE == "run":
    model = joblib.load("gesture_model.pkl")

# Optional: improve resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    # Track the current ML gesture prediction for display and control.
    final_gesture = None

    if result.hand_landmarks:

        hands = result.hand_landmarks
        handedness = result.handedness

        left_hand = None
        right_hand = None

        # 🧠 Assign LEFT and RIGHT hands
        for i in range(len(hands)):
            label = handedness[i][0].category_name

            if label == "Left":
                left_hand = hands[i]
            elif label == "Right":
                right_hand = hands[i]

        # 🎨 Draw BOTH hands
        if left_hand:
            draw_landmarks(frame, left_hand)

        if right_hand:
            draw_landmarks(frame, right_hand)

        # 🏷️ Label hands
        if left_hand:
            x = int(left_hand[0].x * w)
            y = int(left_hand[0].y * h)
            cv2.putText(frame, "LEFT (CONTROL)", (x, y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        if right_hand:
            x = int(right_hand[0].x * w)
            y = int(right_hand[0].y * h)
            cv2.putText(frame, "RIGHT (LOCK)", (x, y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

        # 🔒 LOCK (RIGHT HAND ONLY)
        if right_hand and is_four_fingers(right_hand) and cooldown == 0:
            lock_mode = True
            active_servo = None
            cooldown = 20

        # 🎮 CONTROL (LEFT HAND ONLY)
        if left_hand:
            lm = left_hand

            # Modified Cell 8: build a feature vector with temporal context.
            history.append(lm)
            if len(history) < 5:
                continue

            features = extract_features(lm)
            temporal = extract_temporal_features(history)
            X = np.append(features, temporal)

            # Modified Cell 6: collect labeled samples for training.
            if MODE == "collect":
                label = input("Enter gesture label: ")
                X_data.append(X)
                y_data.append(label)
                final_gesture = label
                print("Saved sample:", label)

            elif MODE == "run":
                gesture = model.predict([X])[0]
                gesture_buffer.append(gesture)

                if len(gesture_buffer) == 5:
                    final_gesture = max(set(gesture_buffer), key=gesture_buffer.count)
                else:
                    final_gesture = gesture

            # Rule-based score/confidence logic removed in favor of classifier labels.
            if MODE == "run" and final_gesture:
                if lock_mode:
                    active_servo = final_gesture
                    lock_mode = False

                if not lock_mode:
                    active_servo = final_gesture

                    if final_gesture == "pinch":
                        control_s1(lm)

                    elif final_gesture == "open":
                        pass  # neutral

                    elif final_gesture == "thumb":
                        control_s2(lm)

                    elif final_gesture == "move":
                        control_s4(lm, w)
                        control_s3(lm)
        else:
            # Reset temporal buffers when the control hand is missing.
            history.clear()
            gesture_buffer.clear()
    else:
        # Reset temporal buffers when no hands are detected.
        history.clear()
        gesture_buffer.clear()

    # ⏳ Cooldown
    if cooldown > 0:
        cooldown -= 1

    # 🖥️ DISPLAY SERVO VALUES
    y0 = 30
    for i, (k, v) in enumerate(servo.items()):
        cv2.putText(frame, f"{k}: {int(v)}", (10, y0 + i*30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

    # STATUS TEXT
    status = "LOCKED" if lock_mode else f"ACTIVE: {active_servo}"
    cv2.putText(frame, status, (10, 180),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    # Modified Cell 9: display the active ML mode and smoothed gesture.
    cv2.putText(frame,
                f"MODE:{MODE} GESTURE:{final_gesture if final_gesture else 'None'} SAMPLES:{len(X_data)}",
                (10, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    cv2.imshow("Gesture Controlled Arm", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# New Cell 9: train and persist the gesture classifier.
if MODE == "train":
    if X_data and y_data:
        model = RandomForestClassifier()
        model.fit(X_data, y_data)
        joblib.dump(model, "gesture_model.pkl")
        print("Model trained and saved")
    else:
        print("No collected samples found. Run collect mode first.")